<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: RPL-1.5

Unless explicitly acquired and licensed from Licensor under another license,
the contents of this file are subject to the Reciprocal Public License ("RPL")
Version 1.5, or subsequent versions as allowed by the RPL, and You may not copy
or use this file in either source code or executable form, except in compliance
with the terms and conditions of the RPL.

All software distributed under the RPL is provided strictly on an "AS IS"
basis, WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESS OR IMPLIED, AND LICENSOR
HEREBY DISCLAIMS ALL SUCH WARRANTIES, INCLUDING WITHOUT LIMITATION, ANY
WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, QUIET
ENJOYMENT, OR NON-INFRINGEMENT. See the RPL for specific language governing
rights and limitations under the RPL.
-->


# Simulation and Intel Tool Runners

This tutorial shows the entry points for plain RTL, Intel component generation, and Platform Designer systems. The cells define reusable launch functions; invoking them requires the corresponding HDL simulator and Intel tools.

## 1. Choosing a runner

| Entry point | Use it for |
|---|---|
| `rtl_test_cocotb()` | Plain Verilog/SystemVerilog source trees |
| `intel_component_test_cocotb()` | A component `.tcl`/`.ip` description requiring `ip-generate` |
| `run_rtl_test()` | A command-line wrapper around the plain RTL runner |
| `run_intel_component_test()` | A command-line wrapper around the Intel component runner |
| `platform_test_cocotb()` | An already generated Platform Designer simulation tree |

All runners use `verilator` by default. Set the `SIM` environment variable to `questa` to select Questa explicitly.

In [1]:
import os
from pathlib import Path

from fpga_verification.sim.runners import rtl_test_cocotb


def run_plain_rtl():
    os.environ["SIM"] = "verilator"
    rtl_test_cocotb(
        project_root=Path("."),
        hdl_toplevel="my_dut",
        test_module="test_my_dut",
        sources=[Path("src/my_dut.sv")],
        parameters={"DATA_WIDTH": 32},
        compile_log=Path("logs/verilator_compile.log"),
    )

# run_plain_rtl()

In [2]:
from fpga_verification.sim.runners import intel_component_test_cocotb


def run_generated_component(generate_only=False):
    os.environ["SIM"] = "verilator"
    return intel_component_test_cocotb(
        project_root=Path("."),
        component_file=Path("src/my_component_hw.tcl"),
        source_dirs=[Path("src/hw")],
        hdl_toplevel="my_component",
        test_module="test_my_component",
        component_parameters={"DATA_WIDTH": 32},
        build_args=["-Wno-PARAMNODEFAULT"],
        generate_only=generate_only,
        ip_generate_log=Path("logs/ip_generate.log"),
        compile_log=Path("logs/verilator_compile.log"),
    )

# generated_directory = run_generated_component(generate_only=True)

In [3]:
from fpga_verification.sim.platform_designer import platform_test_cocotb

def run_generated_platform():
    os.environ["SIM"] = "verilator"
    platform_test_cocotb(
        project_root="platforms/platform_test/sim",
        hdl_toplevel="platform_test",
        test_module="test_platform",
        debug=False,
    )

# run_generated_platform()

## 2. Plain RTL runner

`rtl_test_cocotb()` discovers `.v` and `.sv` files recursively from `source_dirs` when `sources` is omitted. It builds into `sim_build_<SIM>`, forwards HDL parameters, runs the cocotb module, and fails the process when the results XML contains failed tests.

For Questa, `debug=True` enables the GUI, waves, and `wave.do`; `QUESTA_ACC=1` enables signal visibility without full debug mode. For Verilator, the runner supplies the project timescale and C++20 options. `compile_log` redirects compilation output to a file.

## 3. Intel component runner

`intel_component_test_cocotb()` invokes `ip-generate`, reads the generated `.spd` file, combines generated composition HDL with original source files, adds selected Quartus simulation models, and then calls the RTL runner.

Important options include `component_parameters`, `part`, `project_directory`, `quartus_model_files`, `ip_search_paths`, and `make_ipx`. Normally generation uses a temporary directory. `generate_only=True` keeps and returns `.ip_generate_<toplevel>` without simulation; `retain_generated=True` keeps the directory after a normal run.

## 4. Command-line component wrapper

`run_rtl_test()` and `run_intel_component_test()` are intended for small executable `run.py` scripts. Both prepare Python paths and log files, optionally clean the selected simulator build, and accept `-g`. The component wrapper can also derive component parameters from a config object exposing `to_parameters()`. Extra simulator compilation and runtime options can be passed with `build_args` and `test_args`; they are appended to the runner defaults. The wrappers also work when called by pytest because unrelated pytest arguments are ignored. This works for both Verilator and Questa, as selected by `SIM`.

In [4]:
from fpga_verification.sim.runners import run_intel_component_test


def component_run_script():
    run_intel_component_test(
        project_root=Path(__file__).parent,
        component_file="src/my_component_hw.tcl",
        hdl_toplevel="my_component",
        test_module="test_my_component",
        source_dirs=("src",),
        python_paths=(".",),
        component_parameters={"DATA_WIDTH": 32},
        clean_build=True,
    )

`__file__` is available in a saved runner script but not in every notebook environment. Replace it with an explicit project path when experimenting interactively.

## 5. Platform Designer runner

`platform_test_cocotb()` targets an existing generated simulation directory. With Questa it compiles through `msim_setup.tcl` and discovers the generated libraries. With Verilator it extracts a compatible source list from the generated setup. Other `SIM` values are rejected.

Generated Intel trees are tool-version-specific. Keep `project_root`, top-level name, simulator, and Quartus installation consistent, and treat build and generation directories as disposable artifacts rather than source files.